# Phase 3 on Colab — generalisation on s′, the figure, and the demo

Sequencer for [`PHASE3_PLAN.md`](../PHASE3_PLAN.md). The notebook stays thin on purpose: every
number is produced by a tested `embeded/` module, and the cells only order the commands, check
the gates and move artifacts between the VM, the Hugging Face checkpoint and the repo.

> **Update (2026-10-03):** the filtered_* conditions **C4/C5/C6** joined the generalisation design
> (user decision; supersedes the original P6-1 "no fourth strategy" scope for Phase 3). s′ mining now
> produces all six conditions, §8 trains **36** runs (6 × 6 seeds — `SPRIME_SEEDS`, the full Phase-2 ladder, per the 2026-10-03 six-seed decision), and the Δ table/figure cover C0–C6.
>
> **Status (2026-09-28) — this notebook is written ahead of its code.** In the tree today:
> Phase 0/1 and the Phase 2 runner are implemented and unit-tested; the Phase 1 artifacts and all
> nine Phase 2 runs are **still pending**, and `embeded/generalize.py`, `embeded/visualize.py`,
> `demo/app.py` **do not exist yet**. Every Phase 3 cell checks its prerequisite and stops with one
> actionable line instead of a traceback, so the notebook can be reviewed, shared and diffed
> before the code lands. **Nothing here fabricates a number**: a missing module, a missing
> artifact or a failed gate is a stop.

## Order of work
1. T4 runtime → clone the Phase 3 branch → caches, pinned environment, optional HF artifact sync (§0–§1).
2. Preflight: **Phase 2 finished** (six conditions × seeds 13/14/15, plus C0; C4's extension seeds 16–18 exist on HF and are shown as extra evaluations, but are not an entry requirement), artifacts current for `settings.VERSION`, tests green (§2).
3. s′ present and gate-passed *on this machine* — 4,600 pairs / 23 functionalities (§3).
4. Decisions **D3–D6 recorded before any run** (§4), then the s′ census that fixes the holdout (§5).
5. Mine the s′ negatives, then **gate G1s** with the same `hardcheck.evaluate_gap` as G1 — FAIL ⇒ stop (§6).
6. Smoke the s′ runner (§7), then the nine runs (§8) and the free secondary transfer reading (§9).
7. Generalisation table (§10) → **then** the four-panel UMAP figure (§11) → the demo (§12).
8. Review, record, checkpoint, push (§13–§14).

## Rules this notebook cannot enforce for you
- **Nothing is tuned on unseen data.** The threshold comes from the *seen* (validation) part only,
  under the frozen `max_f1_on_valid` policy, and is applied unchanged to the unseen pairs
  (`PHASE3_PLAN.md` §2 item 5).
- **The same three functionalities are held out for every condition and every seed**; §8 re-checks
  it from the run configs, because a per-condition holdout would confound strategy with difficulty.
- Report Δ **per seed and as a mean with the spread visible** — never the best seed, never a
  p-value (SCOPE Part 5).
- **A null result is a finding**: if Δ is indistinguishable across C1/C2/C3, report that. No fourth
  strategy (P6-1), no learning-rate rescue (P6-2), no benchmark switch (P6-5), no added folds to
  quiet the noise (P6-6).
- Quota ladder (P6-8): cut **seeds** before **strategies** (3 → 2 before C3 disappears), and the
  main table wins any competition for GPU time.
- The generalisation runs train on a **different corpus** from the main table (decision D3), so the
  two tables are **not comparable in absolute F1** — say so in both captions.
- The figure is **a figure, not evidence**, and ships *after* the table (FINAL_SPEC correction 8).
  Cite Kitsios et al. (ASE 2025) for the unseen-functionality drop itself: the contribution here is
  only the **strategy × Δ interaction**.

## The CLI this notebook drives (read before running)

`PHASE3_PLAN.md` §6 puts Phase 3 code in `embeded/` (decision D5). The cells below call these
commands from the repo root; they are the contract the modules must satisfy. **If the modules land
with different flag names, update this notebook in the same commit** — a sequencer that guesses is
worse than no sequencer.

| Step | Command | Writes | Gate |
|---|---|---|---|
| census (§5) | `python -m embeded.generalize --census --holdout-k 3` | `artifacts/sprime_census.json` (per-functionality counts + chosen holdout + rule) | counts committed before any F1; rule == D4 |
| mine (§6) | `python -m embeded.generalize --mine --holdout-k 3 --k 20 --cap <settings.SPRIME_TRAIN_CAP>` | `artifacts/sprime_triples_C{1,2,3}.jsonl`, `sprime_mining_summary.json` | same anchors / positives / k in all three conditions |
| hardness (§6) | `python -m embeded.generalize --hardness --holdout-k 3` | `## Gate G1s — hardness on s′` in the report | `embeded.hardcheck.evaluate_gap` — one definition, no second implementation; exit 1 = FAIL ⇒ stop |
| smoke (§7) | `python -m embeded.generalize --condition C1 --seed 13 --holdout-k 3 --smoke` | `artifacts/runs/sprime_smoke_C1_13/` (its own directory) | harness check only |
| run (§8) | `python -m embeded.generalize --condition C1 --seed 13 --holdout-k 3` | `artifacts/runs/sprime_C1_13/{checkpoint.pt, run_config.json, eval_metrics.json, predictions.npz}` | threshold from the seen/validation part only; `run_config.json` records `holdout_functionalities` |
| transfer (§9) | `python -m embeded.generalize --condition C1 --seed 13 --holdout-k 3 --transfer` | `artifacts/runs/sprime_transfer_C1_13/eval_metrics.json` | no training; secondary reading, never the Δ table |
| table (§10) | `python -m embeded.generalize --results-table` | `## Phase 3 — generalisation (F1_seen / F1_unseen / Δ)` in `report/measurements.md` | per-seed rows + mean/spread |
| figure (§11) | `python -m embeded.visualize --panels C0,C1,C2,C3` | `report/figures/umap_panels.png` + `umap_params.json` (seed, `n_neighbors`, `min_dist`, fragment ids) | same fragment sample in every panel |
| demo check (§12) | `python -m demo.app --check` | stdout: the 2 × 2 occupancy of the four outcome modes | exit 0 = every mode has a real held-out pair |
| demo launch (§12) | `python -m demo.app --share` | local Gradio app (Colab prints a share link) | loads checkpoints; never trains, never re-encodes the corpus at import |

Artifacts written by these commands live under `EMBEDED_ARTIFACTS` (outside Git, checkpointed to
the HF dataset repo by §13) except the figure and the report, which are committed.

## 0. Runtime and code checkout

In Colab choose **Runtime → Change runtime type → T4 GPU**. `REPO_REF` below must carry the Phase 3
code; this notebook defaults to the Arena branch it is written on. Use `main` once the Phase 3 work
is merged.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/ksu-gitreaper807/EmbedEd.git'
REPO_REF = 'arena/01a0e714-embeded'   # current Arena work branch (Phase 3 runner + fixes); use main once merged
ROOT = '/content/EmbedEd'

def _git(*args, check=True):
    p = subprocess.run(['git', *args], cwd=ROOT, capture_output=True, text=True)
    if check and p.returncode:
        raise SystemExit(f"git {' '.join(args)} exited {p.returncode}\n{p.stdout}{p.stderr}")
    return p.stdout.strip()

if not os.path.isdir(os.path.join(ROOT, '.git')):
    subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, ROOT],
                   check=True)
else:
    _git('fetch', 'origin', REPO_REF)
    ahead = _git('rev-list', '--count', 'FETCH_HEAD..HEAD', check=False)
    if ahead not in ('', '0'):
        raise SystemExit(
            f'{ahead} local commit(s) on this VM are not on origin/{REPO_REF}. Refusing to move the '
            'branch and orphan them: run section 14 to push them (or `git rebase FETCH_HEAD` by hand), '
            'then re-run this cell.')
    _git('checkout', '-B', REPO_REF, 'FETCH_HEAD')
%cd /content/EmbedEd
print('Checked out:', _git('rev-parse', '--short', 'HEAD'))

## 1. Caches, pinned dependencies, GPU

The same three layers as every session (`COLAB.md`): code from GitHub, packages from the Colab
image, generated artifacts from the Hugging Face dataset repo. The pull is optional — without
`EMBEDED_HF_REPO_ID` / `HF_TOKEN` in Colab Secrets the session starts with local artifacts only.

`torch` is deliberately **not** installed here (Colab's preinstalled build is GPU-matched).
`matplotlib` was added with the Phase 3 figure to `settings.PINNED`, `embeded/requirements.txt`
**and** the `%pip` cell of `notebooks/Phase0_Phase1_Colab.ipynb` — `embeded/tests/test_env_pins.py`
requires those three sources to agree exactly, so the pin belongs here too and is checked in §2.

In [ ]:
import os, sys
ROOT = '/content/EmbedEd'
ART = '/content/embeded-artifacts'
os.makedirs(ART, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf-home'
os.environ['EMBEDED_ARTIFACTS'] = ART
os.environ['EMBEDED_REPORT'] = ROOT + '/report/measurements.md'
try:
    from google.colab import userdata
    for key in ('EMBEDED_HF_REPO_ID', 'EMBEDED_HF_REPO_TYPE', 'EMBEDED_HF_REVISION',
                'EMBEDED_HF_SUBDIR', 'HF_TOKEN'):
        try: value = userdata.get(key)
        except Exception: value = None
        if value: os.environ.setdefault(key, value)
except Exception:
    pass
%pip install -q -U transformers==4.46.3 datasets==2.20.0 sentence-transformers==3.0.1 rank-bm25==0.2.2 umap-learn==0.5.6 scikit-learn==1.6.1 gradio==5.49.1 pytest==8.3.2 matplotlib==3.9.4
import torch, transformers, datasets, sentence_transformers, sklearn, matplotlib
try:
    import umap
    umap_version = umap.__version__
except Exception as exc:   # numba/numpy drift is an environment problem, not a pipeline result:
    umap_version = f'<import failed: {exc.__class__.__name__} - the §11 figure needs it>'
print(f'torch={torch.__version__} cuda={torch.cuda.is_available()} | '
      f'transformers={transformers.__version__} | datasets={datasets.__version__} | '
      f'sentence-transformers={sentence_transformers.__version__} | sklearn={sklearn.__version__} | '
      f'matplotlib={matplotlib.__version__} | umap-learn={umap_version}')
if not torch.cuda.is_available(): raise RuntimeError('GPU is required. Select a Colab T4 runtime before continuing.')
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu)
assert 'T4' in gpu.upper(), f'Expected the measured T4 configuration; got {gpu}. Re-measure throughput before using the frozen budget.'
from scripts.hf_artifacts import main as hf_main
# SECTIONS pull: only what Phase 3 needs (mining inputs, run bookkeeping for the
# preflight, s' triples/census, report) — NOT every checkpoint/npz in the repo.
# Full-run pulls are opt-in where actually used: section 9 (transfer, needs the
# Phase-2 checkpoints) and section 12 (demo). Omitting --section = old behaviour.
hf_main(['pull', '--if-configured', '--light', '--section', 'mining,runs-meta,sprime,report'])

### Runtime helpers

Every later cell uses `sh()`, `need()`, `have_module()`, `hf_pull()` / `hf_push()` and the `S` / `RUNS`
names defined here, with `cwd` moved to the clone. A fresh runtime that starts further down must
re-run this cell first — otherwise `python -m embeded.*` would silently use the **in-repo default
artifacts dir** (`embeded/artifacts/`) instead of the session's.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

ROOT = '/content/EmbedEd'
ART = '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)          # os.chdir alone does not make `embeded` importable
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['PYTHONPATH'] = ROOT + os.pathsep + os.environ.get('PYTHONPATH', '')

from embeded import settings as S
from embeded.train import EVAL_METRICS_NAME, run_dir

RUNS = S.ARTIFACTS / S.RUNS_SUBDIR


def sh(*args, allow_fail=False):
    """`python -m <args>` with stdout/stderr live. The embeded.* runners report most failures as
    SystemExit(message) + exit 1 with no traceback, and a bare CalledProcessError hides the reason."""
    print('$ python -m ' + ' '.join(args), flush=True)
    rc = subprocess.run([sys.executable, '-m', *args]).returncode
    if rc and not allow_fail:
        raise SystemExit(f'\n`python -m {args[0]}` exited {rc} - the runner printed the reason just '
                         'above. If nothing was printed, the three that exit 1 without a traceback '
                         'are: a missing artifact under EMBEDED_ARTIFACTS, a stale mining_summary '
                         'version, and CUDA out of memory.')
    return rc


def need(*paths):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        raise SystemExit('missing prerequisite file(s):\n  ' + '\n  '.join(missing))


def have_module(name):
    from importlib.util import find_spec
    return find_spec(name) is not None


def sprime_dir(condition, seed):
    """Where the generalisation runs live - prefixed, so they can never collide with the Phase 2
    main runs in the same artifacts tree."""
    return RUNS / f'sprime_{condition}_{seed}'


def _hf(argv):
    try:
        from scripts.hf_artifacts import main as hf_main
    except Exception as exc:                            # cell 1 not run in this session
        print('[warn] hf_artifacts unavailable:', exc)
        return 0
    return hf_main(argv)


def hf_pull():
    return _hf(['pull', '--if-configured'])


def hf_push(*extra):
    return _hf(['push', '--if-configured', *extra])


print(f'helpers ready: ROOT={ROOT}\nARTIFACTS={S.ARTIFACTS}\nVERSION={S.VERSION}  '
      f'SEEDS={list(S.SEEDS)}  SPRIME_HELDOUT_K={S.SPRIME_HELDOUT_K}')

## 2. Preflight — Phase 2 must be finished (entry criterion 1)

Phase 3 is *secondary* to the main table: if the two compete for GPU quota, the table wins. The
main table in `report/measurements.md` currently holds one row (C0, F1 0.2572) — this cell refuses
to continue until the trained runs of every main-table condition and the C0 evaluation exist for the current
`settings.VERSION`, so a mis-set `EMBEDED_ARTIFACTS` or a stale checkout surfaces here rather than
40 minutes into a run.

In [ ]:
if 'sh' not in globals():
    raise SystemExit('Run the cells above first: section 1 (caches, pins) and the runtime helpers '
                     'cell. Without them this cell would run `python -m embeded.*` against the '
                     'in-repo default artifacts dir.')

print(f'cwd={Path.cwd()}\nARTIFACTS={S.ARTIFACTS}\nVERSION={S.VERSION}\nREPORT={S.REPORT_MD}')

# 1. Phase 1 artifacts, generated by the checkout this notebook is running ---------------------
required = [S.artifact('fragments.jsonl'), S.artifact('mining_summary.json'),
            *[S.artifact(f'triples_{c}.jsonl') for c in ('C1', 'C2', 'C3', 'C4', 'C5', 'C6')]]
need(*required)
summary = json.loads(S.artifact('mining_summary.json').read_text())
have_version = (summary.get('_run') or {}).get('version')
if have_version != S.VERSION:
    raise SystemExit(
        f'mining_summary.json is version {have_version!r} but settings.VERSION is {S.VERSION!r} - '
        'stale artifacts. Phase 3 does not re-mine the main corpus: back up triples_C*.jsonl, '
        're-mine all six conditions together (`python -m embeded.negatives --strategies '
        'random,bm25,semantic,filtered,filtered_random,filtered_semantic --force`), diff the result against the backup, then re-run this cell. '
        'Identical triples mean the blind audit key still matches its pairs.')

# 2. Phase 2 must be FINISHED (PHASE3_PLAN §2 item 1): every main-table run + the C0 evaluation ----
missing, rows = [], []
# Phase 2's entry inventory: every condition x seeds 13/14/15, symmetric, + the
# C0 evaluation. C4's extension seeds 16-18 exist on HF and are displayed as
# '+ extra' rows below (still scored by section 9's transfer reading), but they
# are NOT an entry requirement — one seed ladder for all six conditions.
REQUIRED = [('C0', S.SEEDS[0])] + [(c, s) for c in ('C1', 'C2', 'C3', 'C4', 'C5', 'C6') for s in S.SEEDS]
for condition, seed in REQUIRED:
    path = run_dir(condition, seed) / EVAL_METRICS_NAME
    if not path.exists():
        missing.append(str(path))
        continue
    m = json.loads(path.read_text())
    if m.get('smoke'):
        missing.append(f'{path}  [smoke run - a harness check, not a result]')
        continue
    rows.append((condition, seed, m.get('version'), float(m['test_f1']), float(m['threshold']),
                 int(m['test']['n'])))
if missing:
    raise SystemExit(
        'Phase 3 is blocked on Phase 2 (PHASE3_PLAN §2 item 1). Missing or invalid:\n  '
        + '\n  '.join(missing)
        + '\nFinish every main-table run (C1-C6 x seeds 13/14/15) and the C0 evaluation with '
          'notebooks/Phase2_Colab.ipynb + Phase2C4_Colab.ipynb + Phase2C5C6_Colab.ipynb (or '
          'scripts/run_all.sh), record the G2 review in PHASE2_PLAN §6, then come back. If the two '
          'compete for GPU quota, the main table wins.')
print(f'{"condition":<9} {"seed":<5} {"version":<11} {"F1":>7} {"thr":>8} {"n":>8}')
for condition, seed, version, f1, thr, n in rows:
    print(f'{condition:<9} {seed:<5} {str(version):<11} {f1:>7.4f} {thr:>8.4f} {n:>8}')
print(f'Phase 2 complete for version {S.VERSION}: {len(rows)} evaluations '
      f'({len(rows) - 1} trained runs + C0).')
shown = {(c, s) for c, s, *_ in rows}
_extra = sorted({(d.name.rpartition('_')[0], int(d.name.rpartition('_')[2]))
                 for d in Path(S.ARTIFACTS / 'runs').glob('C[0-6]_*')
                 if (d / EVAL_METRICS_NAME).exists()} - shown)
for condition, seed in _extra:
    m = json.loads((run_dir(condition, seed) / EVAL_METRICS_NAME).read_text())
    if not m.get('smoke'):
        print(f'+ {condition:<8} {seed:<5} {str(m.get("version")):<11} '
              f'{float(m["test_f1"]):>7.4f} {float(m["threshold"]):>8.4f} {int(m["test"]["n"]):>8}  (extra, not required)')

# 3. Offline correctness before any GPU time ----------------------------------------------------
# the gate stays hard, but a failure must be READABLE: capture, print the tail,
# then stop with a pointer (a bare CalledProcessError helps nobody)
_p = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'embeded/tests'],
                    capture_output=True, text=True)
_out = (_p.stdout or '') + (_p.stderr or '')
print(_out[-4500:])
if _p.returncode:
    raise SystemExit(f'offline test suite FAILED (exit {_p.returncode}) - the failing tests '
                     'are printed above. Do not start Phase 3 on a broken checkout: paste that '
                     'tail into the session. The same commit passes 130/130 on the authoring '
                     'sandbox, so a failure here is checkout- or environment-specific until '
                     'proven otherwise.')

## 3. s′ must be on this machine (entry criterion 2)

RQ3 cannot run without Kitsios et al.'s s′. It is **never pushed to the HF checkpoint**
(`sprime/*` is excluded by `scripts/hf_artifacts.py`), so every fresh VM re-fetches it: completed
files are size- and MD5-verified and skipped, and the probe re-runs cheaply. The gate must print
`s' GATE: PASS` with the same counts measured in Phase 0.7 — 4,600 pairs (2,300 clone + 2,300
non-clone) over 23 functionality ids.

In [ ]:
if 'sh' not in globals():
    raise SystemExit('Run the helper cell in section 1 first.')

rc = sh('scripts.fetch_sprime', allow_fail=True)
if rc:
    raise SystemExit(
        f"`python -m scripts.fetch_sprime` exited {rc}: s' is not on this machine and RQ3 cannot run "
        'without it (PHASE3_PLAN §2 item 2). zenodo.org has to be reachable (the Arena sandbox '
        'blocks it; a Colab VM reaches it fine). If the probe says INCONCLUSIVE, stop and report the '
        'format surprise - do not improvise Route B (hand-labelling) mid-project.')

# Verify the shape directly, not only the exit code: 4,600 pairs = 2,300 + 2,300 over 23
# functionality ids (PROGRESS §2.3).
import pickle, warnings
candidates = sorted(S.artifact('sprime').rglob('*bcb_v2_sampled_bf*.pickle'))
if not candidates:
    raise SystemExit('no *bcb_v2_sampled_bf*.pickle under ' + str(S.artifact('sprime'))
                     + ' - the extraction layout changed; inspect that directory before trusting it.')
frame_path = candidates[0]
with warnings.catch_warnings():
    warnings.simplefilter('ignore', DeprecationWarning)
    with open(frame_path, 'rb') as fh:
        frame = pickle.load(fh)
columns = {str(c) for c in frame.columns}
if not {'code1', 'code2', 'label', 'functionality_id'} <= columns:
    raise SystemExit(f'{frame_path.name}: expected columns code1/code2/label/functionality_id, '
                     f'found {sorted(columns)}')
labels = {int(k): int(v) for k, v in frame['label'].value_counts().items()}
n_func = int(frame['functionality_id'].nunique())
print(f"s' : {frame_path}\n    {len(frame)} pairs, labels {labels}, {n_func} functionalities "
      f"(gate-passed shape: 4,600 = 2,300 + 2,300 over 23)")
if len(frame) != 4600 or n_func != 23:
    raise SystemExit(f"s' shape is not the gate-passed one (4,600 / 23): got {len(frame)} / {n_func}. "
                     'Check settings.SPRIME_DOI and re-download; do not continue on a different corpus.')
hf_push('--include-report')

## 4. Record decisions D3–D6 **before** any run (PHASE3_PLAN §8)

The four open decisions are recorded before the runs and never after seeing the numbers. This cell
writes them to `artifacts/phase3_decisions.json` **and** a dated block in `report/measurements.md`;
every later cell reads the same file, so the commands and the record cannot drift. Defaults are the
plan's recommendations.

- **D3 — where the generalisation negatives come from.** (A) evaluate the thirty-six Phase 2 checkpoints on
  s′ (transfer only, no training); **(B) fine-tune on s′ with the negatives mined from s′ itself** —
  the plan's recommendation and the only option that keeps *negative strategy* as the independent
  variable while training on functionality-labelled data; (C) not possible (CodeXGLUE fragments
  carry no functionality ids). (A) stays available as the free secondary reading in §9.
- **D4 — the "best-represented" rule.** The per-functionality pair counts are **measured** (§5); the
  holdout is the `k` highest counts, ties broken by functionality id, and the rule is fixed before
  any F1 exists.
- **D5 — where Phase 3 code lives.** `embeded/` (`PHASE3_PLAN` header note), matching `CLOUD.md` §4
  and `scripts/run_all.sh`.
- **D6 — what "all four outcome modes" means.** The 2 × 2 of (C0 decision) × (C_best decision), with
  at least one real held-out pair available in the picker for each cell. The phrase is never
  enumerated anywhere in the repo; §12 checks this reading.

In [ ]:
if 'sh' not in globals():
    raise SystemExit('Run the helper cell in section 1 first.')

import re, time

# The plan's recommendations. Change a value only with a written reason in `rationale`, and only
# BEFORE the runs: this block is the anti-hindsight record (PHASE3_PLAN §8).
DECISIONS = {
    'D3': {'choice': 'B',
           'rationale': 'Fine-tune on s′ with negatives mined from s′ itself - the only option that '
                        'keeps negative strategy as the independent variable while training on '
                        'functionality-labelled data (PHASE3_PLAN §3.3). Option (A) is kept as the '
                        'free secondary reading in §9; (C) is impossible on the CodeXGLUE file.'},
    'D4': {'rule': 'max_pair_count_then_lowest_id',
           'rationale': '"Best-represented" is measured, not assumed: the census (§5) prints and '
                        'commits the per-functionality pair counts, takes the k highest and breaks '
                        'ties by functionality id - before any F1 is computed.'},
    'D5': {'choice': 'embeded/',
           'rationale': 'PHASE3_PLAN header note: matches the tested runner modules and both call '
                        'sites (CLOUD.md §4, scripts/run_all.sh); IMPLEMENTATION_PLAN §3 is '
                        're-pointed from eval/ + demo/.'},
    'D6': {'choice': '2x2: (C0 decision) x (C_best decision)',
           'rationale': '"All four outcome modes" (SCOPE P2-21) is never enumerated in the repo; this '
                        'is the reading PHASE3_PLAN §5 asks to confirm, and the demo check in §12 '
                        'requires at least one real held-out pair in the picker for each of the four '
                        'cells.'},
}
AMEND = ''   # a one-line reason; required only if s′ runs already exist (a change after the numbers
             # is an amendment, never an edit - PHASE3_PLAN §8 discipline)

path = S.artifact('phase3_decisions.json')
existing = sorted(p.name for p in RUNS.glob('sprime_*')) if RUNS.exists() else []
if existing and not AMEND.strip():
    raise SystemExit(
        f'{len(existing)} s′ run(s) already exist (e.g. {existing[:3]}). Decisions recorded after the '
        'numbers are hindsight: set AMEND to a one-line reason and re-run. The block is then stamped '
        'as an amendment next to the original instead of silently replacing it (PHASE3_PLAN §8).')

SECTION = '## Phase 3 — decisions (recorded before the runs)'
stamp = time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())
payload = {'recorded_utc': stamp, 'version': S.VERSION, 'holdout_k': S.SPRIME_HELDOUT_K,
           'amendment': AMEND.strip() or None, 'decisions': DECISIONS}
path.write_text(json.dumps(payload, indent=2, ensure_ascii=False))

lines = [SECTION, '',
         f'**recorded {stamp}** · version `{S.VERSION}` · holdout `k = {S.SPRIME_HELDOUT_K}`'
         + (f' · **amendment**: {AMEND.strip()}' if AMEND.strip() else '')
         + ' · source `artifacts/phase3_decisions.json`', '',
         '| id | decision | rationale |', '|---|---|---|']
for key, val in DECISIONS.items():
    lines.append(f"| {key} | {val.get('choice') or val.get('rule')} | {val['rationale']} |")
lines.append('')
block = '\n'.join(lines)

text = S.REPORT_MD.read_text() if S.REPORT_MD.exists() else '# Measurements\n'
if SECTION in text:
    text = re.sub(re.escape(SECTION) + r'.*?(?=\n## |\Z)', lambda _m: block, text, flags=re.S)
else:
    text = text.rstrip('\n') + '\n\n' + block
S.REPORT_MD.write_text(text)
print(json.dumps(payload, indent=2, ensure_ascii=False))
print(f'wrote {path} and "{SECTION}" in {S.REPORT_MD}')

## 5. The s′ census → the holdout is a measured rule (D4)

s′ is described as functionality-*balanced*, but its per-functionality pair counts have never been
recorded in this repo. The census prints them, takes the `k = SPRIME_HELDOUT_K = 3` highest (ties by
functionality id, the rule D4 fixed above), writes `artifacts/sprime_census.json` and appends the
counts to the report. It runs **before** any F1 exists, so the holdout cannot be picked to flatter a
condition.

A 3-of-23 holdout on 4,600 pairs will be noisy, and the audit already puts a 36 % label-noise floor
under the main experiment's negatives. Per SCOPE P6-6: **if the numbers are wildly noisy, report the
noise.** Leave-one-functionality-out rotation is the first SHOULD if time allows — a declared
upgrade, not a rescue.

In [ ]:
if 'sh' not in globals():
    raise SystemExit('Run the helper cell in section 1 first - it defines sh()/have_module().')
if not have_module('embeded.generalize'):
    raise SystemExit(
        'embeded/generalize.py is not in this checkout. It is the Phase 3 code deliverable '
        '(PHASE3_PLAN §6 item 1) and this notebook drives its CLI (contract table at the top of the '
        'file). Land the module with offline tests, then re-run from the top. If the flags differ, '
        'update this notebook in the SAME commit - a sequencer that guesses is worse than none.')

if not S.artifact('phase3_decisions.json').exists():
    raise SystemExit('artifacts/phase3_decisions.json is missing - run the decisions cell (§4) first: '
                     'D4 fixes the holdout rule before the census measures anything, and a decision '
                     'recorded after the numbers are in is hindsight (PHASE3_PLAN §8).')

sh('embeded.generalize', '--census', '--holdout-k', str(S.SPRIME_HELDOUT_K))
census = json.loads(S.artifact('sprime_census.json').read_text())
recorded_d4 = json.loads(S.artifact('phase3_decisions.json').read_text())['decisions']['D4']
if census.get('rule') != recorded_d4.get('rule'):
    raise SystemExit(
        f"the census used holdout rule {census.get('rule')!r} but D4 recorded "
        f"{recorded_d4.get('rule')!r}. The selection rule is fixed before the F1s exist "
        '(PHASE3_PLAN §3.2): fix the module or record an amendment - never let the rule follow the '
        'numbers.')

counts = census.get('counts') or {}
print(f'per-functionality pair counts ({len(counts)} functionalities) - committed to '
      'artifacts/sprime_census.json and the report:')
for fid, n in sorted(counts.items(), key=lambda kv: (-int(kv[1]), str(kv[0])))[:8]:
    print(f'  {str(fid):>6}  {n}')
print(f"... holdout = {census.get('holdout')}  (k={S.SPRIME_HELDOUT_K}, rule "
      f"{census.get('rule')!r}, ties by functionality id)")

## 6. Mine the s′ negatives, then gate G1s — the same gate, one definition

Negatives are mined **inside the 20 held-in functionalities** from s′ itself: C1 a random non-clone
partner, C2 the BM25-nearest non-clone, C3 the cosine-nearest non-clone — and the filtered set
**C4/C5/C6: the same three supplies with the Phase-2 bouncer** (drop J ≥ 0.40, skip 10 survivors) —
the same manipulation as the main experiment, on a corpus that carries functionality labels. The training cap must stay
**smaller** than the main 32,000 and identical across conditions and seeds (`PHASE3_PLAN` §3.4); it
is recorded in `settings.py` next to `SPRIME_HELDOUT_K`.

The hardness verdict is `embeded.hardcheck.evaluate_gap` — **one definition, no second
implementation** — and the `C1 < C2 ≤ C3` ordering verified on CodeXGLUE (`d = 0.614`, gate G1)
**must not be assumed to transfer** to a 4,600-pair corpus. FAIL ⇒ stop and fix the mining
(SCOPE P6-3). Never lower `HARDNESS_D_MIN` to get past it. The gate verdict itself stays the D1 rule
on C1 < C2 ≤ C3 (one definition); C4–C6 are measured and reported alongside it. The strategy set is
fixed at these six — adding a seventh needs a new pre-registration (P6-1's original scope was
superseded on 2026-10-03 when the bouncer factorial joined Phase 3, by user decision).

In [ ]:
if 'sh' not in globals():
    raise SystemExit('Run the helper cell in section 1 first.')

cap = getattr(S, 'SPRIME_TRAIN_CAP', None)
if cap is None:
    raise SystemExit(
        'settings.SPRIME_TRAIN_CAP is not set. PHASE3_PLAN §3.4: the generalisation runs use a cap '
        'SMALLER than the main TRAIN_PAIRS_CAP, recorded in settings.py next to SPRIME_HELDOUT_K and '
        'identical across conditions and seeds. Pick it from the census counts plus a timed s′ smoke '
        'run, add it to settings.py, commit, then re-run this cell. Never normalise the '
        'generalisation runs to the main cap (CLOUD.md §5.6).')
if cap >= S.TRAIN_PAIRS_CAP:
    raise SystemExit(f'SPRIME_TRAIN_CAP={cap} is not smaller than TRAIN_PAIRS_CAP='
                     f'{S.TRAIN_PAIRS_CAP} - the generalisation runs must stay deliberately smaller '
                     '(PHASE3_PLAN §3.4).')

if not S.artifact('sprime_census.json').exists():
    raise SystemExit('artifacts/sprime_census.json is missing - run the census cell (§5) first: the '
                     'holdout has to be a recorded, measured rule before mining trains on it '
                     '(PHASE3_PLAN §3.2).')

sh('embeded.generalize', '--mine', '--holdout-k', str(S.SPRIME_HELDOUT_K),
   '--k', str(S.K_NEGATIVES), '--cap', str(cap))
triples = {c: S.artifact(f'sprime_triples_{c}.jsonl') for c in ('C1', 'C2', 'C3')}
need(*triples.values())
for c, path in triples.items():
    print(f'  {path.name:<28} {sum(1 for _ in open(path, encoding="utf-8"))} triples')

summary = json.loads(S.artifact('sprime_mining_summary.json').read_text())
per_cond = {c: {k: summary[c][k] for k in ('n_anchors', 'k', 'n_triples')
                if isinstance(summary.get(c), dict) and k in summary[c]} for c in ('C1', 'C2', 'C3')}
print('mining summary:', json.dumps(per_cond, indent=2))
shared = [tuple(sorted(v.items())) for v in per_cond.values() if v]
if len(shared) == 3 and len(set(shared)) != 1:
    raise SystemExit('the three conditions disagree on n_anchors/k/n_triples. Shared anchors, '
                     'positives and k are what make "strategy" the only variable (SCOPE P1-3, P2-6): '
                     'fix the mining instead of continuing.')
print('s′ triples mined. Run the G1s cell before any training - a manipulation that did not take '
      'effect produces no evidence (SCOPE P1-6).')
hf_push()

In [ ]:
if 'sh' not in globals():
    raise SystemExit('Run the helper cell in section 1 first.')

rc = sh('embeded.generalize', '--hardness', '--holdout-k', str(S.SPRIME_HELDOUT_K), allow_fail=True)
if rc:
    raise SystemExit(
        f'Gate G1s FAILED (exit {rc}). The manipulation did not take effect on s′ - and it must not '
        'be assumed to transfer from the CodeXGLUE corpus (PHASE3_PLAN §2 item 4). Stop here: fix the '
        's′ mining and re-mine all six conditions together (SCOPE P6-3). An experiment whose '
        'manipulation did not take effect produces no evidence; do not lower HARDNESS_D_MIN (P6-3) '
        'and do not add a seventh condition (the six-condition set is fixed by the 2026-10-03 '
        'decision).')
if '## Gate G1s' not in S.REPORT_MD.read_text():
    raise SystemExit('the gate returned PASS but wrote no "## Gate G1s" section to the report. The '
                     'verdict has to be recorded (PHASE3_PLAN §6 item 1) - fix the module before '
                     'training on it.')
print('Gate G1s PASS - the s′ negatives are ordered C1 < C2 <= C3 with the standardised gap this '
      'corpus actually shows, re-measured rather than assumed.')
hf_push('--include-report')

## 7. Smoke the s′ runner

`--smoke` runs the whole path on a tiny subset — load → forward/backward → checkpoint save/reload →
threshold on the seen/validation part → `F1_seen` / `F1_unseen` — and its numbers mean nothing. It is
the last chance to find a harness error before nine runs. It writes to its OWN directory
(`runs/sprime_smoke_C1_13/`, never `runs/sprime_C1_13/`): a resumed VM re-runs this cell AFTER the
pull has restored finished runs, and when the smoke files shared the real run's directory they
clobbered its metrics — the fingerprint check then demanded a spurious retrain (incident of
2026-09-28). With the separate directory, re-running §7 at any time is always safe.

In [ ]:
if 'sh' not in globals():
    raise SystemExit('Run the helper cell in section 1 first.')

rc = sh('embeded.generalize', '--condition', 'C1', '--seed', str(S.SEEDS[0]),
        '--holdout-k', str(S.SPRIME_HELDOUT_K), '--smoke', allow_fail=True)
if rc:
    raise SystemExit(f'the s′ smoke run exited {rc} - the reason is printed above. Do not start the '
                     'nine runs until this passes. A smoke number is never a result.')
print('smoke OK - harness check only; its metrics are excluded from every table.')

## 8. The thirty-six generalisation runs (6 conditions × 6 seeds)

Fail-fast and checkpointed after every run, exactly like the Phase 2 loop: `embeded.generalize` is
artifact-gated, so a disconnect costs at most one run and re-running this cell resumes. The holdout
identity is re-checked from `run_config.json` after every run, because the same three functionalities
must be held out for **every condition and every seed** — a per-condition holdout would confound the
strategy with the difficulty of the held-out code.

Six seeds per condition (`settings.SPRIME_SEEDS` = 13–18, the 2026-10-03 decision): ≈ 20 min per run at
cap 16,000 → **36 runs ≈ 12 GPU-h**. If quota runs out, the ladder is SCOPE P6-8 as amended: cut
**seeds** first — 6 → 3 (13/14/15, the registered triage core) → 2 — before cutting **conditions**; and
say which ladder step you took in the report — a complete weak comparison beats an incomplete strong
one.

In [ ]:
# stale-kernel guard: the checkout cell updates FILES, but this kernel keeps the
# previously imported modules (Python caches them) - a checkout alone cannot add
# settings attributes to the running session. Restart, don't debug.
if not hasattr(S, 'SPRIME_SEEDS'):
    raise SystemExit('settings.SPRIME_SEEDS is missing: this kernel imported a stale '
                     'checkout. Re-run the checkout cell (section 0), then Runtime -> '
                     'Restart session (files in /content survive), then re-run section 1 '
                     'and this cell. Artifacts (census/mining/gate) are on disk and unaffected.')

# checkpoint-persistence guard: this cell pushes after EVERY run; with HF
# unconfigured those pushes silently skip (by design) and an interrupted
# campaign would leave everything only on this VM's disk.
import os
if not os.environ.get('EMBEDED_HF_REPO_ID') and os.environ.get('EMBEDED_ALLOW_NO_HF', '') != '1':
    raise SystemExit('EMBEDED_HF_REPO_ID is not set: every per-run checkpoint push below would '
                     'silently skip, so an interruption would leave the runs only on this VM. '
                     'Fix: add/enable the Colab secrets (EMBEDED_HF_REPO_ID, EMBEDED_HF_REPO_TYPE, '
                     'EMBEDED_HF_REVISION, EMBEDED_HF_SUBDIR, HF_TOKEN) with notebook access, '
                     're-run the section-1 env cell, and confirm the pull probe prints "[hf] '
                     'pulled N files". To run deliberately without sync, set '
                     "os.environ['EMBEDED_ALLOW_NO_HF'] = '1' in the env cell and note it in the "
                     'report.')

if 'sh' not in globals():
    raise SystemExit('Run the helper cell in section 1 first.')

import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU: the generalisation runs need the T4. Change the runtime type first.')
cap = getattr(S, 'SPRIME_TRAIN_CAP', None)          # set in settings.py; the mining cell (§6) explains how
census_path, summary_path = S.artifact('sprime_census.json'), S.artifact('sprime_mining_summary.json')
need(census_path, summary_path, *[S.artifact(f'sprime_triples_{c}.jsonl') for c in ('C1', 'C2', 'C3', 'C4', 'C5', 'C6')])
if '## Gate G1s' not in S.REPORT_MD.read_text():
    raise SystemExit('gate G1s has not been recorded in the report - run §6 (mine, then the gate) '
                     'before training: an experiment whose manipulation did not take effect produces '
                     'no evidence (SCOPE P1-6, P6-3).')
holdout = sorted(json.loads(census_path.read_text()).get('holdout') or [])
if len(holdout) != S.SPRIME_HELDOUT_K:
    raise SystemExit(f'the census records {len(holdout)} held-out functionalities, but '
                     f'SPRIME_HELDOUT_K is {S.SPRIME_HELDOUT_K}: re-run the census (§5) before '
                     'training.')
print(f'holdout in force: {holdout} (k={S.SPRIME_HELDOUT_K}); cap={cap}; '
      f'GPU={torch.cuda.get_device_name(0)}')

runs = [(c, s) for c in ('C1', 'C2', 'C3', 'C4', 'C5', 'C6') for s in S.SPRIME_SEEDS]
for i, (condition, seed) in enumerate(runs, 1):
    print(f'\n===== {i}/{len(runs)} GENERALISE {condition} seed={seed} =====', flush=True)
    rc = sh('embeded.generalize', '--condition', condition, '--seed', str(seed),
            '--holdout-k', str(S.SPRIME_HELDOUT_K), allow_fail=True)
    if rc:
        raise SystemExit(f'{condition} seed {seed} exited {rc} - the reason is printed above. Fix it, '
                         'or apply the P6-8 seed ladder deliberately and say so in the report; do not '
                         'quietly skip a condition and let the comparison look complete.')
    cfg_path = sprime_dir(condition, seed) / 'run_config.json'
    if cfg_path.exists():
        got = sorted(json.loads(cfg_path.read_text()).get('holdout_functionalities') or [])
        if got != holdout:
            raise SystemExit(f'{cfg_path} trained with holdout {got}, not {holdout}: a different '
                             'holdout per run is a confound, not a result (PHASE3_PLAN §3.2). Stop '
                             'and fix the runner.')
    hf_push()                                       # checkpoint each completed run
print(f'\nAll {len(runs)} generalisation runs finished for version {S.VERSION}.')

## 9. Free secondary reading — D3 option (A), clearly **not** the Δ table

Scoring the thirty-six Phase 2 checkpoints — plus the untuned C0 base model once, on the same seen/unseen
split — costs no training and answers a different question: how does a CodeXGLUE-trained model
transfer to s′? It **cannot** enter the Δ table, because it never trains on functionality-labelled
data — the independent variable would be gone. C0 goes through the same `--transfer` flag
(`--condition C0 --seed 13`), lands in `runs/sprime_transfer_C0_13/`, and is the table's baseline
row. Record the block as a secondary observation only, and label it that way in the write-up.

In [ ]:
if not hasattr(S, 'SPRIME_SEEDS'):
    raise SystemExit('stale kernel: re-run the checkout cell, restart the session, re-run section 1.')
if 'sh' not in globals():
    raise SystemExit('Run the helper cell in section 1 first.')
# this reading scores the PHASE-2 checkpoints -> it needs the full run dirs
# (checkpoints included, ~4.5 GB): opt-in here, not in section 1.
hf_main(['pull', '--if-configured', '--section', 'runs'])


# C0 first: the untuned baseline needs no checkpoint and is evaluated once (seed 13),
# exactly like the Phase 2 C0 - it becomes the baseline row of the §10 table.
runs = [('C0', S.SEEDS[0])] + [(c, s) for c in ('C1', 'C2', 'C3', 'C4', 'C5', 'C6') for s in S.SPRIME_SEEDS]
for condition, seed in runs:
    rc = sh('embeded.generalize', '--condition', condition, '--seed', str(seed),
            '--holdout-k', str(S.SPRIME_HELDOUT_K), '--transfer', allow_fail=True)
    if rc:
        print(f'[optional] transfer reading for {condition} seed {seed} unavailable (exit {rc}) - '
              'skipped; this reading is secondary by definition (PHASE3_PLAN §3.3).')
hf_push()
print('Transfer readings live in artifacts/runs/sprime_transfer_* - they are never part of the Δ table.')

## 10. The generalisation table — `F1_seen` / `F1_unseen` / Δ

The second of `SCOPE.md` P2-19's three tables.

- `F1_seen` — pairs whose functionality was trained on, measured on the held-in slice the frozen seed
  keeps out of training (never on the training pairs themselves).
- `F1_unseen` — the three held-out functionalities, untouched by training, mining and threshold
  selection.
- Δ = `F1_seen` − `F1_unseen` **per condition, per seed**, then the mean with the spread visible.

The threshold comes from the *seen* validation part only and is applied unchanged; the table shows it
so a reader can see it was not tuned on the unseen pairs. C0 is evaluated once on the same partition
through the same scoring path, and the caption must say that this table's corpus differs from the
main table's (D3), so absolute F1 is **not** comparable across the two.

In [ ]:
if 'sh' not in globals():
    raise SystemExit('Run the helper cell in section 1 first.')

sh('embeded.generalize', '--results-table')
text = S.REPORT_MD.read_text()
if '## Phase 3 — generalisation' not in text:
    raise SystemExit('no "## Phase 3 — generalisation" section in the report after --results-table: '
                     'the module has to write it (PHASE3_PLAN §6 item 2).')
block = text.split('## Phase 3 — generalisation', 1)[1].split('\n## ', 1)[0]
for token in ('F1_seen', 'F1_unseen', 'Δ'):
    if token not in block:
        raise SystemExit(f'the generalisation table never mentions {token!r} - it is incomplete.')
print('## Phase 3 — generalisation' + block)

# Holdout identity across all nine runs: the same three functionalities, every condition and seed.
configs = {}
for condition in ('C1', 'C2', 'C3', 'C4', 'C5', 'C6'):
    for seed in S.SEEDS:
        path = sprime_dir(condition, seed) / 'run_config.json'
        if path.exists():
            configs[f'{condition}_{seed}'] = sorted(
                json.loads(path.read_text()).get('holdout_functionalities') or [])
if not configs:
    raise SystemExit('no artifacts/runs/sprime_*_*/run_config.json found - there is nothing to '
                     'summarise yet: run the generalisation runs (§8) first.')
if len(configs) < 36:
    print(f'[warn] only {len(configs)} of 36 run configs found - the table is partial (and a partial '
          'table may only be reported if the P6-8 seed ladder was taken deliberately).')
distinct = {tuple(v) for v in configs.values()}
if len(distinct) > 1:
    raise SystemExit(f'the runs do not share one holdout: {configs} - strategy would be confounded '
                     'with holdout difficulty (PHASE3_PLAN §3.2). Fix the runner and re-run before '
                     'reporting Δ.')
print(f'holdout verified identical across {len(configs)} run configs: {sorted(distinct)[0]}')
hf_push('--include-report')

## 11. The seven-panel UMAP figure — **after** the table

Seven panels — C0–C6 — with a **fixed seed** and fixed `n_neighbors` / `min_dist` both
recorded in `settings.py` (SCOPE P2-18), **the same sample of fragments in every panel**, coloured by
functionality *including the held-out ones*, so the unseen-functionality story is visible in the same
picture as the Δ table.

This is a figure, not evidence (FINAL_SPEC correction 8): it ships after the table, and no claim in
the write-up may rest on it. `report/figures/` is generated, never hand-edited — the cell below
refuses to run until the §10 table exists.

In [ ]:
if 'sh' not in globals():
    raise SystemExit('Run the helper cell in section 1 first.')

if '## Phase 3 — generalisation' not in S.REPORT_MD.read_text():
    raise SystemExit('the generalisation table is not in the report yet - the figure ships AFTER the '
                     'table (FINAL_SPEC correction 8; PHASE3_PLAN §4). Run §10 first.')
sh('embeded.visualize', '--panels', 'C0,C1,C2,C3,C4,C5,C6')
fig_dir = Path(ROOT) / 'report' / 'figures'
need(fig_dir / 'umap_panels.png', fig_dir / 'umap_params.json')
params = json.loads((fig_dir / 'umap_params.json').read_text())
print(json.dumps(params, indent=2)[:2000])
for key in ('seed', 'n_neighbors', 'min_dist', 'fragment_ids'):
    if key not in params:
        raise SystemExit(f'umap_params.json has no {key!r}: the figure must record its seed, its '
                         'hyperparameters and the exact fragment sample (SCOPE P2-18, PHASE3_PLAN §4).')
print('figure written; it is a figure, not evidence - it is presented after the tables.')
try:                                              # Colab only; elsewhere the file is already local
    from google.colab import files
    files.download(str(fig_dir / 'umap_panels.png'))
except Exception as exc:
    print(f'(download skipped: {exc.__class__.__name__}) - the figure is at '
          f'{fig_dir / "umap_panels.png"}')

## 12. The demo — real held-out pairs, all four outcome modes

`demo/app.py`: one file, Gradio, no serving infrastructure (SCOPE Part 5 — the demo is a local
single-file app, not a deployment). Default view: **real held-out test pairs with their gold
labels** (not synthetic examples, not a blank box). Per pair: `gold` / `C0 cosine vs its threshold` /
`C_best cosine vs its threshold`, with the threshold drawn as a **visible marker on the score bar**,
so the reader can see *why* the decision came out as it did. Free-text paste stays, labelled
*"illustrative only — not part of the evaluation"*, in those words.

`--check` is headless: it loads the committed run metadata plus `predictions.npz` / `checkpoint.pt`,
never trains, never re-encodes the corpus at import, and asserts that each of the four cells in D6's
2 × 2 has at least one real pair in the picker. Checkpoints live outside Git, so the pull below is
the demo's first action.

In [ ]:
if 'sh' not in globals():
    raise SystemExit('Run the helper cell in section 1 first.')

hf_main(['pull', '--if-configured', '--section', 'runs'])   # demo loads checkpoints +
    # predictions.npz; they live outside Git (full runs section: checkpoints INCLUDED)
rc = sh('demo.app', '--check', allow_fail=True)
if rc:
    raise SystemExit(
        f'the demo check exited {rc}. The app must load only committed run metadata + predictions.npz '
        '/ checkpoint.pt (no training, no corpus re-encode at import), show real held-out pairs by '
        'default with their gold labels and both thresholds, and reach every one of D6\'s four '
        'outcome modes (PHASE3_PLAN §5, SCOPE P2-21). Fix it before shipping the demo.')
print('demo check OK - all four outcome modes have real held-out pairs; the free-text box is labelled '
      '"illustrative only - not part of the evaluation".')

In [ ]:
if 'sh' not in globals():
    raise SystemExit('Run the helper cell in section 1 first.')

# Optional: the demo is a local single-file Gradio app, not a deployment. On Colab `--share` prints
# a public URL and blocks this cell; interrupt the cell to stop the server.
sh('demo.app', '--share')

## 13. Review, record, checkpoint

Before writing anything up:

- **Every implausible number is a possible harness error first** — pair order, labels, threshold
  handling, which pairs count as "seen" — never a result. That is the G3-style review entry
  `PHASE3_PLAN` §6 item 6 requires, and it goes in the plan's §9 log with the command that produced
  the number.
- Verify the artifacts: nine s′ runs plus the C0 evaluation, per-seed `F1_seen` / `F1_unseen` / Δ,
  thresholds recorded, holdout identical across runs (the §10 cell already asserts the last one).
- With §10 the **three tables of SCOPE P2-19** are all present: main benchmark (Phase 2),
  generalisation (§10), measured false-negative rates (Phase 2 audit: C2 42 %, C3 30 %, pooled 36 %).
- Keep the **label-noise floor** (the 36 % pooled false-negative rate on the CodeXGLUE-mined
  negatives) and s′'s **gold labels** separate in the write-up — they are different measurements on
  different corpora.
- Append what was actually done to `PHASE3_PLAN.md` §9 (command, artifacts, outcome, deviations),
  and note that `scripts/run_all.sh` still ends at `# Phase 3+ (TODO)` and must be extended past it
  (PHASE3_PLAN §6 item 5).
- Say it in both captions: the generalisation table trains on a **different corpus** (D3), so it is
  not comparable in absolute F1 to the main table.

In [ ]:
hf_push('--include-report')
import shutil
shutil.make_archive('/content/phase3_report', 'zip', Path(ROOT) / 'report')
try:
    from google.colab import files
    files.download('/content/phase3_report.zip')
except Exception as exc:
    print(f'(download skipped: {exc.__class__.__name__}) - report/ is also in the HF checkpoint and '
          'in the VM clone.')
print('report zipped: /content/phase3_report.zip (includes report/figures/).')

## 14. Push the VM's commits back to GitHub

Same shape as the Phase 2 notebook's §8, condensed. Three separate things make `git push` fail on a
Colab VM and they all surface as **exit 128**: no commit identity, a dirty worktree, and no
credential helper with no terminal to prompt on. `scripts.colab_git` handles all three (it refuses
to stage `audit_key*`, redacts every byte of git output, and keeps the token out of `.git/config` and
`argv`); the full walk-through — including recovery from a clone stuck mid-rebase — is in `COLAB.md`.
The token goes in Colab's **Secrets** panel as `GITHUB_TOKEN`, never in cell source.

In [ ]:
import json as _json, os, subprocess, sys, urllib.request
if 'REPO_REF' not in globals():
    raise SystemExit('Run cell 0 first: it defines ROOT and REPO_REF.')
ROOT, REF = '/content/EmbedEd', REPO_REF

def git(*args, check=False):
    env = {k: v for k, v in os.environ.items()
           if not k.startswith(('GIT_AUTHOR_', 'GIT_COMMITTER_'))}   # these override user.email
    p = subprocess.run(['git', '-C', ROOT, *args], capture_output=True, text=True, env=env)
    shown = (p.stdout + p.stderr).strip()
    print(f"$ git {' '.join(args)} -> exit {p.returncode}" + (f'\n  {shown}' if shown else ''))
    if check and p.returncode:
        raise SystemExit(f'git {" ".join(args)} failed - stopping, nothing was lost')
    return p.returncode, p.stdout

sys.path.insert(0, ROOT)
from scripts.colab_git import read_secret

# 1. identity first: every commit and every rebase needs one, and a fresh VM has none
if not git('config', '--get', 'user.email')[1].strip():
    token = read_secret(('GITHUB_TOKEN',))
    if not token:
        raise SystemExit('No git identity and no GITHUB_TOKEN in Colab Secrets. Add the fine-grained '
                         'PAT as a Secret named GITHUB_TOKEN (or set user.name/user.email by hand), '
                         'then re-run this cell - nothing was changed.')
    req = urllib.request.Request('https://api.github.com/user',
                                 headers={'Authorization': f'Bearer {token}',
                                          'Accept': 'application/vnd.github+json',
                                          'User-Agent': 'embeded-colab'})
    with urllib.request.urlopen(req, timeout=20) as resp:
        info = _json.loads(resp.read().decode())
    git('config', 'user.name', info['login'], check=True)
    git('config', 'user.email', f"{info['id']}+{info['login']}@users.noreply.github.com", check=True)
    print(f"  identity derived from the PAT: {info['login']}")

# 2. commit tracked report/ changes (figures included) and refuse anything else; audit_key* is
#    refused by colab_git itself, so the blind key can never be pushed
_, raw = git('status', '--porcelain=v1')
tracked = [line[3:].strip() for line in raw.splitlines()
           if line.strip() and line[:2].strip() != '??']
strays = [path for path in tracked if not path.startswith('report/')]
if strays:
    raise SystemExit(f'tracked changes outside report/: {strays} - commit or stash them, then re-run.')
if tracked:
    git('add', '--', *tracked, check=True)
    git('commit', '-m', 'report: Phase 3 generalisation, figure and demo check', check=True)

# 3. rebase onto the branch tip, then push through the helper that redacts git output
git('fetch', 'origin', REF, check=True)
if git('rebase', 'FETCH_HEAD')[0]:
    raise SystemExit('rebase stopped - your commits are safe. Resolve and `git rebase --continue`, or '
                     '`git -C /content/EmbedEd rebase --abort`; see COLAB.md, "Recovering a clone '
                     'stuck mid-rebase".')
sys.modules.pop('scripts.colab_git', None)      # a cached pre-rebase copy would shadow the new code
from scripts.colab_git import diagnose, read_secret, sync_and_push

print('\n--- diagnose ---')
for line in diagnose(ROOT, REF):
    print(line)
token = read_secret(('GITHUB_TOKEN',))
if not token:
    print('No GITHUB_TOKEN in Colab Secrets -> stopping before the push. The identity, the commit and '
          'the rebase are done; add the Secret and re-run this cell to push.')
else:
    print('\n--- push ---')
    result = sync_and_push(ROOT, REF, token=token)
    for line in result['lines']:
        print(line)
    print(('ok  ' if result['ok'] else 'FAIL') + f" [{result['stage']}] {result['message']}")